In [2]:
import scanpy as sc
import sys

pancreas_path = "../data/raw/Pancreas_TSP1_30_version2d_10X_smartseq_scvi_Nov122024.h5ad"
ovary_path   = "../data/raw/Ovary_TSP1_30_version2d_10X_smartseq_scvi_Nov262024.h5ad"
v1_codes = [f"TSP{i}" for i in range(1, 16)]
v2_codes = ["TSP17", "TSP19", "TSP20", "TSP21", "TSP25", "TSP26", "TSP27", "TSP28", "TSP30"]

adata = sc.read_h5ad(ovary_path)

2. (Optional) Visualize Data

In [3]:
donors_in_data = set(adata.obs['donor'].astype(str).unique())

found     = sorted([d for d in v2_codes if d in donors_in_data])
not_found = sorted([d for d in v2_codes if d not in donors_in_data])

print(f"Total new donors to check : {len(v2_codes)}")
print(f"Donors present in data   : {len(found)} -> {found}")
print(f"Donors NOT present       : {len(not_found)} -> {not_found}")


Total new donors to check : 9
Donors present in data   : 3 -> ['TSP27', 'TSP28', 'TSP30']
Donors NOT present       : 6 -> ['TSP17', 'TSP19', 'TSP20', 'TSP21', 'TSP25', 'TSP26']


In [4]:
print(adata.var.head())
print(adata.obs.head())

                  ensembl_id gene_symbol       genome     mt   ercc  \
index                                                                 
TSPAN6    ENSG00000000003.15      TSPAN6  Gencode_v41  False  False   
TNMD       ENSG00000000005.6        TNMD  Gencode_v41  False  False   
DPM1      ENSG00000000419.14        DPM1  Gencode_v41  False  False   
SCYL3     ENSG00000000457.14       SCYL3  Gencode_v41  False  False   
C1orf112  ENSG00000000460.17    C1orf112  Gencode_v41  False  False   

          n_cells_by_counts  mean_counts  pct_dropout_by_counts  total_counts  \
index                                                                           
TSPAN6               161872     2.379617              91.800375     4697694.0   
TNMD                   9323     0.220273              99.527743      434850.0   
DPM1                 461590     3.523875              76.618161     6956619.0   
SCYL3                156149     0.493041              92.090273      973332.0   
C1orf112        

In [ ]:

print(adata.obs.columns.tolist())


print(adata.obs[['cell_ontology_id', 'free_annotation', 'manually_annotated']].head())

for col in ['cell_ontology_id', 'free_annotation', 'manually_annotated']:
    non_null = adata.obs[col].notna().sum()
    total    = adata.n_obs
    print(f"{col}: {non_null}/{total} cells annotated")

['donor', 'tissue', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'old_index', 'assay', 'sample_id', 'replicate', '10X_run', '10X_barcode', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_ontology_class', 'cell_ontology_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'age', 'sex', 'ethnicity', 'scvi_leiden_res05_tissue', 'sample_number']
                                               cell_ontology_id  \
TSP30_Ovary_NA_SS3_Blue_B105390_LiveDead_D9          CL:0000192   
TSP30_Ovary_NA_SS3_Blue_B105394_LiveDead_E22         CL:0000115   
TSP30_Ovary_NA_SS3_Blue_B105394_LiveDead_P4          CL:0000192   
TSP30_Ovary_NA_SS3_B004411_B105393_LiveDead_I1       CL:0000192   
TSP30_Ovary_N

3. Filter by new donors

In [3]:
mask = adata.obs['donor'].isin(v2_codes) & ~adata.obs['donor'].isin(v1_codes)
adata = adata[mask].copy()

4. Hide labels

In [4]:
adata_test = adata.copy()                     
adata_test.obs.drop(columns=[
    'cell_ontology_id',
    'free_annotation',
    'manually_annotated'
], inplace=True)
true_labels = adata.obs['cell_ontology_id'].copy()

5. Hyperparamter Settings


In [5]:
hyperparams = {
    "seed": 42,                        
    "dataset_name": "ovary_v2",        
    "load_model": "../models/scGPT_human",
    "do_train": True,                  
    "mask_ratio": 0.0,                 
    "n_bins": 51,                      
    "epochs": 10,                      
    "batch_size": 64,                  
    "lr": 1e-4,                        
    "layer_size": 128,                 
    "nlayers": 4,                      
    "nhead": 4,                        
    "dropout": 0.2,                    
    "save_eval_interval": 2,           
    "amp": True,                       
    "freeze": False,                   
}

6. Preprocess & Bin

In [1]:
import torch
print("torch:", torch.__version__)
import torchtext
print("torchtext:", torchtext.__version__)



torch: 2.3.0+cu121
torchtext: 0.18.0+cpu
